# Advanced Module 5 · Multi-Agent Orchestration
Connecting the agents from Module 4 — an orchestrator that routes, dispatches in parallel,
runs debate/consensus, and handles failures gracefully.

---
> **Prerequisite:** Complete Module 4 (Real-World Agents) — we reuse those agent definitions here.  
> **Setup:** Run the first two cells before anything else.

In [1]:
%pip install -q google-genai groq python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, time, re, getpass, random, threading, string
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv
from google import genai
from google.genai import types
from groq import Groq

load_dotenv()
api_key  = os.environ.get('GEMINI_API_KEY') or getpass.getpass('Paste your Gemini API key: ')
groq_key = (os.environ.get('Groq_key') or getpass.getpass('Paste your Groq API key: ')).strip()

client      = genai.Client(api_key=api_key)
groq_client = Groq(api_key=groq_key)
MODEL       = 'gemini-3.1-flash-lite'
GROQ_FAST   = 'llama-3.1-8b-instant'
GROQ_REASON = 'qwen/qwen3-32b'
DEFAULT_MAX = 2048

def cfg(**kwargs):
    kwargs.setdefault('max_output_tokens', DEFAULT_MAX)
    kwargs.setdefault('temperature', 0.7)
    return types.GenerateContentConfig(**kwargs)

def get_text(response) -> str:
    if response.text:
        return response.text.strip()
    parts = []
    for cand in (response.candidates or []):
        if cand.content:
            for part in cand.content.parts:
                if not getattr(part, 'thought', False) and part.text:
                    parts.append(part.text)
    return ''.join(parts).strip()

def _call_with_retry(fn, *args, max_retries=5, **kwargs):
    for attempt in range(max_retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            msg = str(e)
            if '429' in msg or 'RESOURCE_EXHAUSTED' in msg:
                m = re.search(r'retry[^0-9]*([0-9]+)s', msg, re.I)
                wait = int(m.group(1)) + 5 if m else 35
                print(f'  ⏳ Rate-limited — waiting {wait}s')
                time.sleep(wait)
            elif '500' in msg or 'INTERNAL' in msg:
                time.sleep(10 * (attempt + 1))
            else:
                raise
    raise RuntimeError('Max retries exceeded')

_raw_gen = client.models.generate_content
client.models.generate_content = lambda *a, **kw: _call_with_retry(_raw_gen, *a, **kw)

print('✅ Setup complete | Gemini:', MODEL)

✅ Setup complete | Gemini: gemini-3.1-flash-lite


In [3]:
try:
    _r = client.models.generate_content(model=MODEL,
        contents='Reply with exactly the words: Hello LLM course', config=cfg(temperature=0.0))
    print('✅ API working:', get_text(_r))
except Exception as e:
    print('❌ Error:', e)

✅ API working: Hello LLM course


---
## The Architecture: One Orchestrator, Many Workers

Each Module 4 agent is now a **worker** — a callable black box that takes a task and returns a result.  
The **orchestrator** is a new LLM call that: classifies the user's intent, decides which worker(s) to call,  
sequences or parallelizes them, handles failures, and synthesizes the final response.

```
┌──────────────────────────────────────────────────────────────────┐
│                    MULTI-AGENT ARCHITECTURE                      │
│                                                                  │
│  User Request: "Help me plan a business trip and check if my     │
│                order ORD-1002 will arrive before I leave."       │
│                         │                                        │
│                         ▼                                        │
│              ┌──────────────────────┐                            │
│              │    ORCHESTRATOR      │  ← classifies intent,      │
│              │   (Gemini / Groq)    │    decides routing         │
│              └──────────┬───────────┘                            │
│                         │                                        │
│           ┌─────────────┼─────────────┐                          │
│           ▼             ▼             ▼                          │
│   ┌──────────────┐ ┌──────────┐ ┌──────────────┐                │
│   │   Booking    │ │ Support  │ │  Research    │                │
│   │    Agent     │ │  Agent   │ │   Agent      │                │
│   └──────┬───────┘ └────┬─────┘ └──────┬───────┘                │
│          │              │              │                         │
│          └──────────────┴──────────────┘                         │
│                         │                                        │
│                         ▼                                        │
│              ┌──────────────────────┐                            │
│              │    ORCHESTRATOR      │  ← merges results,         │
│              │  (synthesis step)    │    writes final answer     │
│              └──────────────────────┘                            │
└──────────────────────────────────────────────────────────────────┘
```

In [4]:
# ── Rebuild the 3 worker agents from Module 4 as callable functions ──

# --- Booking Agent tools ---
def search_flights(origin, destination, date):
    return {'route': f'{origin}→{destination}', 'date': date, 'options': [
        {'flight_id':'SK201','airline':'SkyWings', 'price_usd':580,'duration_h':9},
        {'flight_id':'SW900','airline':'SwiftAir', 'price_usd':520,'duration_h':10},
        {'flight_id':'LX101','airline':'LuxAir',   'price_usd':1200,'duration_h':8},
    ]}
def search_hotels(city, checkin, checkout):
    return {'city':city,'checkin':checkin,'checkout':checkout,'options':[
        {'hotel_id':'H01','name':'CityInn',       'price_per_night_usd':85, 'rating':3.8,'stars':3},
        {'hotel_id':'H02','name':'ParkView Grand','price_per_night_usd':140,'rating':4.5,'stars':4},
        {'hotel_id':'H04','name':'Royal Suite',   'price_per_night_usd':350,'rating':4.9,'stars':5},
    ]}
def book_itinerary(flight_id, hotel_id, passenger_name):
    ref = ''.join(random.choices(string.ascii_uppercase+string.digits, k=8))
    prices_f = {'SK201':580,'SW900':520,'LX101':1200}
    prices_h = {'H01':85,'H02':140,'H04':350}
    return {'booking_ref':ref,'status':'CONFIRMED','passenger':passenger_name,
            'flight_cost':prices_f.get(flight_id,0),'hotel_cost':prices_h.get(hotel_id,0)}
def get_exchange_rate(from_currency, to_currency):
    rates={'USD':1.0,'EUR':0.93,'GBP':0.79,'INR':83.5}
    r=rates.get(to_currency.upper(),1.0)/rates.get(from_currency.upper(),1.0)
    return {'from':from_currency,'to':to_currency,'rate':round(r,4)}

# --- Support Agent tools ---
def lookup_order(order_id):
    db={'ORD-1001':{'status':'delivered','item':'Laptop Pro 15"','price':1299,'delivery_date':'2025-07-10'},
        'ORD-1002':{'status':'in_transit','item':'Wireless Headphones','price':249,'eta':'2025-07-20'},
        'ORD-1003':{'status':'cancelled','item':'Smart Watch','price':399}}
    return db.get(order_id,{'error':f'Order {order_id} not found'})
def check_refund_policy(reason):
    p={'defective':{'eligible':True,'method':'full refund'},'not_needed':{'eligible':True,'method':'refund -10%'},
       'late':{'eligible':True,'method':'full refund'},'other':{'eligible':False,'method':'escalate'}}
    return p.get(reason.lower(),p['other'])
def raise_ticket(order_id, issue_type, description):
    return {'ticket_id':f'TKT-{random.randint(10000,99999)}','status':'open','sla_hours':24}
def escalate_to_human(reason, order_id):
    return {'escalated':True,'queue':'tier-2','wait_min':15,
            'message':f'Escalated {order_id} for: {reason}'}

# --- Research Agent tools ---
ARTICLE_DB = {
    'https://techblog.example.com/ai-agents-2025': {'title':'AI Agents 2025',
        'text':'AI agents handle 40% of support queries. Multi-agent systems show 3x productivity gains. Hallucination rates 8-12%.'},
    'https://research.example.com/llm-productivity': {'title':'LLM Productivity',
        'text':'Developers 55% faster with AI assistants. Code review time down 30%. 25% of startup codebases AI-generated.'},
    'https://safety.example.com/agent-risks': {'title':'Agent Safety Risks',
        'text':'Prompt injection on 15 production systems. Unrestricted agents caused $2.3M costs. Allowlists recommended.'},
}
def list_available_articles(topic):
    return {'topic':topic,'urls':list(ARTICLE_DB.keys())}
def fetch_article(url):
    a=ARTICLE_DB.get(url); return {'url':url,'title':a['title'],'content':a['text']} if a else {'error':'not found'}
def extract_key_claims(text):
    resp=groq_client.chat.completions.create(model=GROQ_FAST,
        messages=[{'role':'user','content':f'Extract 2-3 key factual claims as JSON array:\n{text}'}],
        max_tokens=200, temperature=0.0)
    content=resp.choices[0].message.content
    try:
        m=re.search(r'\[.*?\]',content,re.DOTALL); claims=json.loads(m.group(0)) if m else [content]
    except: claims=[content]
    return {'claims':claims}
def fact_check(claim):
    resp=groq_client.chat.completions.create(model=GROQ_FAST,
        messages=[{'role':'user','content':f'Rate as LIKELY_TRUE/UNCERTAIN/LIKELY_FALSE with reason. Claim: {claim}\nJSON: {{"verdict":"...","reason":"..."}}'}],
        max_tokens=100, temperature=0.0)
    content=resp.choices[0].message.content
    try:
        m=re.search(r'\{.*?\}',content,re.DOTALL); r=json.loads(m.group(0)) if m else {'verdict':'UNCERTAIN','reason':content}
    except: r={'verdict':'UNCERTAIN','reason':content[:80]}
    return {**r,'claim':claim}

# --- Assemble tool schemas and maps ---
from google.genai import types as gt

def _str(desc=''): return gt.Schema(type=gt.Type.STRING, description=desc)

BOOKING_SCHEMA = gt.Tool(function_declarations=[
    gt.FunctionDeclaration(name='search_flights', description='Search available flights.',
        parameters=gt.Schema(type=gt.Type.OBJECT, required=['origin','destination','date'],
            properties={'origin':_str('Departure city'),'destination':_str('Arrival city'),'date':_str('YYYY-MM-DD')})),
    gt.FunctionDeclaration(name='search_hotels', description='Search hotels in a city.',
        parameters=gt.Schema(type=gt.Type.OBJECT, required=['city','checkin','checkout'],
            properties={'city':_str(),'checkin':_str('YYYY-MM-DD'),'checkout':_str('YYYY-MM-DD')})),
    gt.FunctionDeclaration(name='book_itinerary', description='Confirm flight and hotel booking.',
        parameters=gt.Schema(type=gt.Type.OBJECT, required=['flight_id','hotel_id','passenger_name'],
            properties={'flight_id':_str(),'hotel_id':_str(),'passenger_name':_str()})),
    gt.FunctionDeclaration(name='get_exchange_rate', description='Get exchange rate between currencies.',
        parameters=gt.Schema(type=gt.Type.OBJECT, required=['from_currency','to_currency'],
            properties={'from_currency':_str(),'to_currency':_str()})),
])
BOOKING_MAP = {'search_flights':search_flights,'search_hotels':search_hotels,
               'book_itinerary':book_itinerary,'get_exchange_rate':get_exchange_rate}

SUPPORT_SCHEMA = gt.Tool(function_declarations=[
    gt.FunctionDeclaration(name='lookup_order', description='Look up order by ID.',
        parameters=gt.Schema(type=gt.Type.OBJECT, required=['order_id'], properties={'order_id':_str()})),
    gt.FunctionDeclaration(name='check_refund_policy', description='Check refund policy for a reason.',
        parameters=gt.Schema(type=gt.Type.OBJECT, required=['reason'], properties={'reason':_str()})),
    gt.FunctionDeclaration(name='raise_ticket', description='Create a support ticket.',
        parameters=gt.Schema(type=gt.Type.OBJECT, required=['order_id','issue_type','description'],
            properties={'order_id':_str(),'issue_type':_str(),'description':_str()})),
    gt.FunctionDeclaration(name='escalate_to_human', description='Escalate issue to a human agent.',
        parameters=gt.Schema(type=gt.Type.OBJECT, required=['reason','order_id'],
            properties={'reason':_str(),'order_id':_str()})),
])
SUPPORT_MAP = {'lookup_order':lookup_order,'check_refund_policy':check_refund_policy,
               'raise_ticket':raise_ticket,'escalate_to_human':escalate_to_human}

RESEARCH_SCHEMA = gt.Tool(function_declarations=[
    gt.FunctionDeclaration(name='list_available_articles', description='List available article URLs.',
        parameters=gt.Schema(type=gt.Type.OBJECT, required=['topic'], properties={'topic':_str()})),
    gt.FunctionDeclaration(name='fetch_article', description='Fetch article text by URL.',
        parameters=gt.Schema(type=gt.Type.OBJECT, required=['url'], properties={'url':_str()})),
    gt.FunctionDeclaration(name='extract_key_claims', description='Extract key claims from text.',
        parameters=gt.Schema(type=gt.Type.OBJECT, required=['text'], properties={'text':_str()})),
    gt.FunctionDeclaration(name='fact_check', description='Fact-check a single claim.',
        parameters=gt.Schema(type=gt.Type.OBJECT, required=['claim'], properties={'claim':_str()})),
])
RESEARCH_MAP = {'list_available_articles':list_available_articles,'fetch_article':fetch_article,
                'extract_key_claims':extract_key_claims,'fact_check':fact_check}

print('✅ All 3 worker agents rebuilt')

✅ All 3 worker agents rebuilt


In [5]:
# Generic agent runner (same as Module 4 — copied here for self-containedness)
def run_agent(task, system_prompt, tool_schema, tool_map, max_steps=10, label='Agent', verbose=True):
    conversation = [gt.Content(role='user', parts=[gt.Part(
        text=f'[SYSTEM]: {system_prompt}\n\n[TASK]: {task}'
    )])]
    total_tokens = 0
    if verbose:
        print(f'\n  🤖 {label} started: {task[:80]}...' if len(task)>80 else f'\n  🤖 {label}: {task}')

    for step in range(1, max_steps+1):
        resp = client.models.generate_content(
            model=MODEL, contents=conversation,
            config=cfg(tools=[tool_schema], temperature=0.2)
        )
        total_tokens += resp.usage_metadata.total_token_count
        fcs = [p.function_call for p in resp.candidates[0].content.parts
               if hasattr(p,'function_call') and p.function_call]
        if not fcs:
            final = get_text(resp)
            if verbose:
                print(f'  [Step {step}] ✅ Done | Tokens: {total_tokens}')
            return final
        results = []
        for fc in fcs:
            try:
                r = tool_map[fc.name](**dict(fc.args))
            except Exception as e:
                r = {'error': str(e)}
            results.append(r)
            if verbose:
                print(f'  [Step {step}] 🔧 {fc.name}({dict(fc.args)}) → {str(r)[:80]}')
        conversation.append(resp.candidates[0].content)  # preserves thought_signature
        conversation.append(gt.Content(role='user', parts=[
            gt.Part(function_response=gt.FunctionResponse(name=fc.name, response={'result':r}))
            for fc, r in zip(fcs, results)
        ]))
    return 'Max steps reached.'

print('✅ Agent runner ready')

✅ Agent runner ready


---
## Demo 1: Orchestrator Routing

An ambiguous user message — the orchestrator classifies intent and decides which agent(s) to invoke.

In [6]:
AGENT_REGISTRY = {
    'booking_agent': {
        'description': 'Handles flight search, hotel search, booking, and trip cost calculations.',
        'schema': BOOKING_SCHEMA, 'map': BOOKING_MAP,
        'system': 'You are a smart travel booking assistant. Search flights and hotels, then book the best option.'
    },
    'support_agent': {
        'description': 'Handles customer support: order lookup, refunds, tickets, and escalations.',
        'schema': SUPPORT_SCHEMA, 'map': SUPPORT_MAP,
        'system': 'You are a customer support agent. Be empathetic. Look up orders. Escalate complex cases.'
    },
    'research_agent': {
        'description': 'Fetches articles, extracts claims, fact-checks them, and writes research reports.',
        'schema': RESEARCH_SCHEMA, 'map': RESEARCH_MAP,
        'system': 'You are a research analyst. Fetch articles, extract claims, fact-check, write structured reports.'
    },
}

def orchestrate(user_request: str, verbose: bool = True) -> str:
    """Step 1 — Orchestrator classifies intent and selects agents to run."""

    agent_descriptions = '\n'.join(
        f'- {name}: {meta["description"]}' for name, meta in AGENT_REGISTRY.items()
    )

    routing_prompt = f"""You are an orchestrator. Given a user request, output a JSON object:
{{'agents': [list of agent names to invoke], 'subtasks': {{agent_name: subtask_for_that_agent}}}}

Available agents:
{agent_descriptions}

User request: {user_request}

Rules:
- Only include agents that are actually needed
- If multiple agents are needed, list all of them
- The subtask should be a concise task description for that agent
- Return ONLY the JSON, no explanation"""

    routing_resp = client.models.generate_content(
        model=MODEL,
        contents=routing_prompt,
        config=cfg(temperature=0.0)
    )
    routing_text = get_text(routing_resp)

    # Parse routing decision
    try:
        match   = re.search(r'\{.*\}', routing_text, re.DOTALL)
        routing = json.loads(match.group(0)) if match else {'agents': [], 'subtasks': {}}
    except Exception:
        routing = {'agents': [], 'subtasks': {}}

    if verbose:
        print('═' * 65)
        print(f'USER: {user_request}')
        print('═' * 65)
        print(f'\n🎯 ORCHESTRATOR ROUTING DECISION:')
        print(f'   Agents selected: {routing.get("agents", [])}')
        for agent, subtask in routing.get('subtasks', {}).items():
            print(f'   {agent} → "{subtask}"')

    # Run the selected agents sequentially
    agent_results = {}
    for agent_name in routing.get('agents', []):
        if agent_name not in AGENT_REGISTRY:
            continue
        meta      = AGENT_REGISTRY[agent_name]
        subtask   = routing.get('subtasks', {}).get(agent_name, user_request)
        result    = run_agent(
            task=subtask, system_prompt=meta['system'],
            tool_schema=meta['schema'], tool_map=meta['map'],
            label=agent_name, verbose=verbose
        )
        agent_results[agent_name] = result

    # Synthesize
    if not agent_results:
        return 'No agents were invoked. Please rephrase your request.'

    synthesis_prompt = (
        f'Original user request: {user_request}\n\n'
        + '\n\n'.join(f'Output from {name}:\n{result}' for name, result in agent_results.items())
        + '\n\nSynthesize a clear, complete final response to the user.'
    )
    final = client.models.generate_content(
        model=MODEL, contents=synthesis_prompt, config=cfg(temperature=0.3)
    )
    final_text = get_text(final)

    if verbose:
        print('\n' + '═' * 65)
        print('🏁 ORCHESTRATOR FINAL RESPONSE:')
        print('═' * 65)
        print(final_text)

    return final_text

# Run it
orchestrate(
    "Help me plan a business trip from Mumbai to Berlin next month. "
    "Also, check what's happening with my order ORD-1002."
)

═════════════════════════════════════════════════════════════════
USER: Help me plan a business trip from Mumbai to Berlin next month. Also, check what's happening with my order ORD-1002.
═════════════════════════════════════════════════════════════════

🎯 ORCHESTRATOR ROUTING DECISION:
   Agents selected: ['booking_agent', 'support_agent']
   booking_agent → "Plan a business trip from Mumbai to Berlin for next month, including flight and hotel options."
   support_agent → "Look up the status of order ORD-1002."

  🤖 booking_agent started: Plan a business trip from Mumbai to Berlin for next month, including flight and ...
  [Step 1] 🔧 search_flights({'destination': 'Berlin', 'date': '2025-06-15', 'origin': 'Mumbai'}) → {'route': 'Mumbai→Berlin', 'date': '2025-06-15', 'options': [{'flight_id': 'SK20
  [Step 2] 🔧 search_hotels({'city': 'Berlin', 'checkout': '2025-06-20', 'checkin': '2025-06-16'}) → {'city': 'Berlin', 'checkin': '2025-06-16', 'checkout': '2025-06-20', 'options':
  [Step 3

'Here is the complete update regarding your business trip and your recent order:\n\n### **Business Trip: Mumbai to Berlin (June 15th – June 20th)**\nI have researched flight and hotel options for your upcoming trip. Based on a balance of cost, travel time, and professional amenities, I recommend the following:\n\n*   **Flight:** SkyWings (SK201) – $580 (9-hour duration)\n*   **Hotel:** ParkView Grand – $140/night (4.5-star rating, ideal for business)\n\n**Other available options:**\n*   **Flights:** LuxAir ($1,200, 8 hours) or SwiftAir ($520, 10 hours).\n*   **Hotels:** Royal Suite ($350/night, luxury) or CityInn ($85/night, budget).\n\n**Next Steps:** Would you like me to proceed with booking the recommended combination? If so, please provide the passenger name to finalize the reservation.\n\n***\n\n### **Order Status: ORD-1002**\nRegarding your order for the Wireless Headphones, I have confirmed that the package is currently in transit. It is expected to arrive by **July 20, 2025**.\

---
## Demo 2: Parallel Worker Dispatch

When two agents can run independently, dispatch them in parallel using threads.  
Print the wall-clock time saved vs. running them sequentially.

In [7]:
def run_parallel_agents(tasks: dict, verbose: bool = True) -> dict:
    """
    tasks = {'agent_name': 'task string', ...}
    Returns {agent_name: result} after running all agents in parallel.
    """
    results = {}

    def run_one(agent_name, task):
        meta = AGENT_REGISTRY[agent_name]
        return agent_name, run_agent(
            task=task, system_prompt=meta['system'],
            tool_schema=meta['schema'], tool_map=meta['map'],
            label=agent_name, verbose=verbose
        )

    with ThreadPoolExecutor(max_workers=len(tasks)) as executor:
        futures = {executor.submit(run_one, name, task): name for name, task in tasks.items()}
        for future in as_completed(futures):
            name, result = future.result()
            results[name] = result

    return results

# ── Comparison: sequential vs. parallel ──
task_a = 'I want to fly from Delhi to Tokyo on 2025-09-01 for 2 nights. Find cheapest options.'
task_b = 'Research the current state of AI agents in 2025. Summarize key findings.'

print('Running agents SEQUENTIALLY...')
t0 = time.time()
res_a = run_agent(task_a, AGENT_REGISTRY['booking_agent']['system'],
                  BOOKING_SCHEMA, BOOKING_MAP, label='booking_agent', verbose=False)
res_b = run_agent(task_b, AGENT_REGISTRY['research_agent']['system'],
                  RESEARCH_SCHEMA, RESEARCH_MAP, label='research_agent', verbose=False, max_steps=8)
seq_time = round(time.time() - t0, 2)
print(f'Sequential time: {seq_time}s')

print('\nRunning agents IN PARALLEL...')
t0 = time.time()
parallel_results = run_parallel_agents({
    'booking_agent':  task_a,
    'research_agent': task_b,
}, verbose=True)
par_time = round(time.time() - t0, 2)
print(f'\nParallel time: {par_time}s')

print('\n' + '=' * 65)
print('TIMING COMPARISON')
print(f'  Sequential : {seq_time}s')
print(f'  Parallel   : {par_time}s')
speedup = round(seq_time / max(par_time, 0.1), 2)
print(f'  Speedup    : {speedup}x faster')
print(f'  Saved      : {round(seq_time - par_time, 2)}s')
print('\nParallel dispatch is the key to scalable multi-agent systems.')

Running agents SEQUENTIALLY...
  ⏳ Rate-limited — waiting 17s
  ⏳ Rate-limited — waiting 60s
Sequential time: 87.1s

Running agents IN PARALLEL...

  🤖 booking_agent started: I want to fly from Delhi to Tokyo on 2025-09-01 for 2 nights. Find cheapest opti...

  🤖 research_agent: Research the current state of AI agents in 2025. Summarize key findings.
  [Step 1] 🔧 search_flights({'date': '2025-09-01', 'origin': 'Delhi', 'destination': 'Tokyo'}) → {'route': 'Delhi→Tokyo', 'date': '2025-09-01', 'options': [{'flight_id': 'SK201'
  [Step 1] 🔧 list_available_articles({'topic': 'AI agents 2025 state of the art'}) → {'topic': 'AI agents 2025 state of the art', 'urls': ['https://techblog.example.
  [Step 2] 🔧 search_hotels({'checkout': '2025-09-03', 'city': 'Tokyo', 'checkin': '2025-09-01'}) → {'city': 'Tokyo', 'checkin': '2025-09-01', 'checkout': '2025-09-03', 'options': 
  [Step 2] 🔧 fetch_article({'url': 'https://techblog.example.com/ai-agents-2025'}) → {'url': 'https://techblog.example.com/

---
## Demo 3: Debate + Consensus (with Reasoning Model as Judge)

Two instances of the Research Agent argue opposite sides of a decision.  
A Judge agent (Qwen3 reasoning model) weighs both arguments and picks the stronger one —  
showing the thinking trace.

This pattern is used in: agentic code review, investment analysis, risk assessment.

In [8]:
DEBATE_TOPIC = "For a 500-person engineering org, should we adopt microservices or stick with a monolith?"

# Two adversarial agents with opposing system prompts
PRO_MICRO_PROMPT = (
    'You are arguing STRONGLY IN FAVOR of microservices architecture. '
    'Make the best possible case with concrete benefits, scalability advantages, and industry examples. '
    'Do NOT mention any downsides.'
)
PRO_MONO_PROMPT = (
    'You are arguing STRONGLY IN FAVOR of a monolith architecture. '
    'Make the best possible case: simplicity, operational ease, faster development, and real-world successes. '
    'Do NOT mention any downsides.'
)

print('═' * 65)
print('DEBATE TOPIC:', DEBATE_TOPIC)
print('═' * 65)

# Run both debaters in parallel
micro_arg, mono_arg = None, None

def get_micro_arg():
    global micro_arg
    resp = client.models.generate_content(
        model=MODEL,
        contents=f'[SYSTEM]: {PRO_MICRO_PROMPT}\n\n[TOPIC]: {DEBATE_TOPIC}',
        config=cfg(temperature=0.7, max_output_tokens=512)
    )
    micro_arg = get_text(resp)

def get_mono_arg():
    global mono_arg
    resp = client.models.generate_content(
        model=MODEL,
        contents=f'[SYSTEM]: {PRO_MONO_PROMPT}\n\n[TOPIC]: {DEBATE_TOPIC}',
        config=cfg(temperature=0.7, max_output_tokens=512)
    )
    mono_arg = get_text(resp)

t1 = threading.Thread(target=get_micro_arg)
t2 = threading.Thread(target=get_mono_arg)
t1.start(); t2.start()
t1.join();  t2.join()

print('\n📌 ARGUMENT FOR MICROSERVICES:')
print(micro_arg[:500], '...' if len(micro_arg) > 500 else '')
print('\n📌 ARGUMENT FOR MONOLITH:')
print(mono_arg[:500], '...' if len(mono_arg) > 500 else '')

# Qwen3 reasoning model as Judge
print('\n' + '─' * 65)
print('🧠 JUDGE (Qwen3-32B reasoning model) deliberating...')
print('─' * 65)

judge_prompt = f"""You are a neutral technical judge evaluating two arguments about architecture choice.

TOPIC: {DEBATE_TOPIC}

ARGUMENT A (Microservices):
{micro_arg}

ARGUMENT B (Monolith):
{mono_arg}

Evaluate both arguments. Which is stronger given the context (500-person eng org)?
Provide:
1. Your verdict (A or B) with detailed reasoning
2. The key strengths and weaknesses of each argument
3. A nuanced recommendation for the org"""

judge_resp = groq_client.chat.completions.create(
    model=GROQ_REASON,
    messages=[{'role': 'user', 'content': judge_prompt}],
    max_tokens=2048,
    temperature=0.6
)

judge_content = judge_resp.choices[0].message.content
think_match   = re.search(r'<think>(.*?)</think>', judge_content, re.DOTALL)
thinking      = think_match.group(1).strip() if think_match else ''
verdict       = re.sub(r'<think>.*?</think>', '', judge_content, flags=re.DOTALL).strip()

if thinking:
    print('\n[JUDGE THINKING TRACE (first 400 chars)]:')
    print(thinking[:400], '...')

print('\n[JUDGE VERDICT]:')
print(verdict)

═════════════════════════════════════════════════════════════════
DEBATE TOPIC: For a 500-person engineering org, should we adopt microservices or stick with a monolith?
═════════════════════════════════════════════════════════════════

📌 ARGUMENT FOR MICROSERVICES:
For an engineering organization of 500 people, the transition to a microservices architecture is not merely an architectural choice—it is a strategic imperative for survival and market dominance. At this scale, a monolith becomes a structural bottleneck that actively suppresses velocity, innovation, and reliability.

Moving to microservices is the only way to unlock the true potential of a 500-person engineering force. Here is the case for why microservices are the definitive path forward.

###  ...

📌 ARGUMENT FOR MONOLITH:
For an engineering organization of 500 people, the answer is clear: **stick with the monolith.** The industry has been misled by the "microservices hype cycle," but the reality is that the monolith is t

---
## Demo 4: Failure & Fallback — Resilient Orchestration

The Booking Agent hits a simulated tool failure mid-run.  
The orchestrator detects it and re-routes to the Support Agent as a fallback.

In [9]:
booking_fail_count = {'n': 0}

def search_flights_failing(origin, destination, date):
    """Fails on every call to simulate an outage."""
    raise ConnectionError('OUTAGE: Flight search service is down. Please try again later or contact support.')

FAILING_BOOKING_MAP = {**BOOKING_MAP, 'search_flights': search_flights_failing}

def orchestrate_with_fallback(user_request: str):
    print('═' * 65)
    print(f'USER: {user_request}')
    print('═' * 65)

    # Try booking agent first
    print('\n🎯 Routing to: booking_agent')
    try:
        result = run_agent(
            task=user_request,
            system_prompt=AGENT_REGISTRY['booking_agent']['system'],
            tool_schema=BOOKING_SCHEMA,
            tool_map=FAILING_BOOKING_MAP,  # ← the failing version
            label='booking_agent (primary)',
            verbose=True,
            max_steps=3  # fail fast
        )

        # Check if result contains an error signal
        if 'Max steps' in result or 'OUTAGE' in result or 'error' in result.lower():
            raise RuntimeError(f'Booking agent failed: {result[:100]}')

        return result

    except Exception as e:
        print(f'\n⚠️  PRIMARY AGENT FAILED: {e}')
        print('🔄 ORCHESTRATOR: Falling back to support_agent...')

        fallback_task = (
            f'The user tried to make a booking but our flight search system is down. '
            f'Original request: "{user_request}". '
            f'Please apologize professionally, raise a support ticket, '
            f'and let the user know we will follow up within 24 hours.'
        )
        fallback_result = run_agent(
            task=fallback_task,
            system_prompt=AGENT_REGISTRY['support_agent']['system'],
            tool_schema=SUPPORT_SCHEMA,
            tool_map={**SUPPORT_MAP, 'lookup_order': lambda order_id: {'status': 'N/A', 'note': 'system outage'}},
            label='support_agent (fallback)',
            verbose=True
        )

        print('\n' + '═' * 65)
        print('🏁 FALLBACK RESPONSE TO USER:')
        print('═' * 65)
        print(fallback_result)
        return fallback_result

orchestrate_with_fallback(
    "I need to book a flight from Chennai to Singapore for next week, 4 nights."
)

═════════════════════════════════════════════════════════════════
USER: I need to book a flight from Chennai to Singapore for next week, 4 nights.
═════════════════════════════════════════════════════════════════

🎯 Routing to: booking_agent

  🤖 booking_agent (primary): I need to book a flight from Chennai to Singapore for next week, 4 nights.
  [Step 1] 🔧 search_flights({'date': '2025-05-19', 'origin': 'Chennai', 'destination': 'Singapore'}) → {'error': 'OUTAGE: Flight search service is down. Please try again later or cont
  [Step 2] 🔧 search_hotels({'checkout': '2025-05-23', 'checkin': '2025-05-19', 'city': 'Singapore'}) → {'city': 'Singapore', 'checkin': '2025-05-19', 'checkout': '2025-05-23', 'option
  [Step 3] ✅ Done | Tokens: 1701


"It seems that the flight search service is currently experiencing an outage, so I cannot retrieve flight options for you at the moment.\n\nHowever, I have found some hotel options in Singapore for your 4-night stay (May 19th – May 23rd):\n\n*   **CityInn**: $85/night (3 stars, 3.8 rating)\n*   **ParkView Grand**: $140/night (4 stars, 4.5 rating)\n*   **Royal Suite**: $350/night (5 stars, 4.9 rating)\n\nWould you like me to keep monitoring the flight service and let you know once it's back up, or would you like to proceed with a hotel booking for now?"

In [10]:
# ✏️  Add a 4th custom agent to the AGENT_REGISTRY and plug it into the orchestrator.
#
#    Ideas:
#    - WeatherAgent: check weather forecast for a city using a stub tool
#    - CurrencyAgent: compare exchange rates across multiple currencies
#    - VisaAgent: check visa requirements between two countries
#
#    Steps:
#    1. Define tool function(s)
#    2. Build tool schema using gt.FunctionDeclaration
#    3. Add to AGENT_REGISTRY with a name, description, schema, map, and system prompt
#    4. Re-run orchestrate() with a task that would trigger your new agent

# Example structure:
# AGENT_REGISTRY['weather_agent'] = {
#     'description': 'Checks weather forecast for any city.',
#     'schema': my_weather_schema,
#     'map':    {'get_forecast': my_get_forecast},
#     'system': 'You are a weather information agent. Use the forecast tool to answer weather questions.'
# }

print('Define your agent, add it to AGENT_REGISTRY, then run:')
print('orchestrate("Ask something that requires your new agent")')

Define your agent, add it to AGENT_REGISTRY, then run:
orchestrate("Ask something that requires your new agent")


---
## Full-Day Architecture Diagram

```
┌─────────────────────────────────────────────────────────────────────┐
│                  EVERYTHING YOU BUILT TODAY                         │
│                                                                     │
│  MODULE 1: Function Calling                                         │
│  ┌──────┐  tool JSON   ┌─────────┐  result   ┌───────┐             │
│  │ LLM  │────────────▶│ Python  │──────────▶│  LLM  │             │
│  └──────┘              └─────────┘           └───────┘             │
│                                                                     │
│  MODULE 2: ReAct Agent Loop                                         │
│  [Think] → [Act] → [Observe] → [Think] → ... → [Answer]            │
│                                                                     │
│  MODULE 3: MCP + Reasoning + Safety                                 │
│  MCP: standard protocol for tool discovery and invocation           │
│  Reasoning: <think>...</think> for hard problems                    │
│  Safety: allowlists stop prompt injection attacks                   │
│                                                                     │
│  MODULE 4: Specialized Workers                                      │
│  BookingAgent  ──┐                                                  │
│  SupportAgent  ──┤  (each = system prompt + tools + ReAct loop)     │
│  ResearchAgent ──┘                                                  │
│                                                                     │
│  MODULE 5: Orchestration                                            │
│                      ┌──────────────┐                               │
│                      │ ORCHESTRATOR │  classify → route → merge     │
│                      └──────┬───────┘                               │
│          ┌───────────────────┼──────────────────┐                   │
│          ▼                   ▼                  ▼                   │
│   BookingAgent        SupportAgent        ResearchAgent             │
│   (parallel ←──────────────────────────────────→ parallel)         │
│          │                   │                  │                   │
│          └───────────────────┴──────────────────┘                   │
│                              │                                      │
│                      ┌───────▼───────┐                              │
│                      │  SYNTHESIS    │  merge all outputs           │
│                      └───────────────┘                              │
│                                                                     │
│  Patterns used: routing, parallel dispatch, debate+consensus,       │
│                 failure+fallback, reasoning model as judge          │
└─────────────────────────────────────────────────────────────────────┘
```

---
## Key Takeaways

| Concept | What it means |
|---|---|
| Orchestrator | An LLM that classifies user intent and decides which worker agents to run |
| Worker agents | Specialized agents from Module 4 — callable black boxes |
| Routing | Orchestrator outputs a JSON `{agents: [...], subtasks: {...}}` decision |
| Parallel dispatch | Run independent agents in threads — drastically reduces wall-clock time |
| Debate + consensus | Run adversarial agents on opposite sides, use reasoning model as judge |
| Failure + fallback | Detect agent failures, re-route to a fallback agent — resilient systems |
| Synthesis | Final orchestrator call merges all worker outputs into a coherent response |

---
## Full-Day Summary Cheat Sheet

| Module | Key idea | When to use |
|---|---|---|
| 1 — Function Calling | LLM returns JSON; your code runs the function | Any time the model needs to access external data or systems |
| 2 — ReAct Loop | Reason → Act → Observe until done | Open-ended tasks where the number of steps is unknown in advance |
| 3 — MCP | Standard protocol; any client works with any server | Building tools that need to be reusable across models and frameworks |
| 3 — Reasoning | `<think>` trace; slower but better on hard problems | Math, logic puzzles, ambiguous instructions, complex decisions |
| 3 — Safety | Allowlist every tool call before executing | Any agent that acts on real systems — mandatory, not optional |
| 4 — Real Agents | System prompt + tools + ReAct = a complete agent | Any domain-specific automation task |
| 5 — Orchestration | One orchestrator + N workers = scalable multi-agent system | Complex tasks that need multiple specializations or parallelism |